# Simple LLM Application with LCEL

This notebook builds a small translation / text-generation app using **LangChain Expression Language (LCEL)** with a model served by **Groq**.

**What's covered:**
1. Loading API keys from a `.env` file
2. Creating a chat model (`ChatGroq`)
3. Calling the model directly with `SystemMessage` / `HumanMessage`
4. Cleaning the model output with `StrOutputParser`
5. Chaining components together with the LCEL pipe operator (`|`)
6. Making the input reusable with `ChatPromptTemplate`
7. A full `prompt | model | parser` chain, plus an article-writing example

**Prerequisites:** a `.env` file in the project folder containing `GROQ_API_KEY`, `LANGCHAIN_API_KEY`, `LANGCHAIN_PROJECT` (and `OPENAI_API_KEY` if used), plus the packages `langchain`, `langchain-groq` and `python-dotenv`.

---

## Step 1: Load environment variables
`load_dotenv()` reads the key/value pairs from the `.env` file and puts them into the process environment so `os.getenv(...)` can find them.

> ⚠️ Note: the call needs parentheses — `load_dotenv()`. Writing just `load_dotenv` only references the function and does **not** load the file.

In [37]:
import os
from dotenv import load_dotenv
load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: Union[str, ForwardRef('os.PathLike[str]'), NoneType] = None, stream: Optional[IO[str]] = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: Optional[str] = 'utf-8') -> bool>

## Step 2: Set API keys and enable LangSmith tracing
Copy the keys from `.env` into `os.environ` so the LangChain libraries can pick them up automatically.

- `LANGCHAIN_TRACING_V2 = "true"` turns on **LangSmith** tracing — every chain run is logged to the LangSmith dashboard under the project named in `LANGCHAIN_PROJECT`, which is useful for debugging.
- `GROQ_API_KEY` is the key used by the Groq chat model below.

In [38]:
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ['LANGCHAIN_TRACING_V2'] = "true"
os.environ['LANGCHAIN_PROJECT'] = os.getenv("LANGCHAIN_PROJECT")
os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

## Step 3: Initialize the chat model
`ChatGroq` is LangChain's wrapper around models hosted on Groq. Here we use the open-weights `openai/gpt-oss-20b` model.

Displaying `model` shows its configuration (model name, token limits, capabilities, etc.).

In [39]:
from langchain_groq import ChatGroq
import os

model = ChatGroq(
    model="openai/gpt-oss-20b",
    groq_api_key=os.getenv("GROQ_API_KEY")
)

model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.4', 'langchain': '1.5.2', 'langchain-groq': '1.2.1'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2025-08-05', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001E7BDE43210>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E7BE2F9690>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

## Step 4: Call the model directly with messages
Chat models take a list of messages:
- **`SystemMessage`** – instructions that set the model's behaviour (here: translate English → German).
- **`HumanMessage`** – the actual user input to act on.

`model.invoke(messages)` returns an `AIMessage` object; the generated text lives in `response.content`.

In [40]:
from langchain_core.messages import HumanMessage,SystemMessage

messages = [
    SystemMessage(
        content = "Translate the following from English to german"
        ),
    HumanMessage(
        content = "Hello How are you?"
        )
]

response = model.invoke(messages)

In [41]:
print(response.content)

Hallo, wie geht es dir?


## Step 5: Extract plain text with an output parser
`StrOutputParser` takes the model's `AIMessage` and returns just the text as a plain Python string. This saves us from writing `response.content` everywhere and lets the parser be plugged into a chain.

In [42]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
parser.invoke(response)

'Hallo, wie geht es dir?'

## Step 6: Chain components with LCEL
**LCEL (LangChain Expression Language)** lets you connect components using the pipe operator `|`. The output of the left side becomes the input of the right side:

```
messages ──► model ──► parser ──► string
```

`chain = model | parser` does in one call what Steps 4 and 5 did separately.

In [43]:
## USing the LCEL - chain the components

chain = model | parser
chain.invoke(messages)

'Hallo, wie geht es dir?'

## Step 7: Make the input reusable with a prompt template
Hard-coding messages isn't flexible. `ChatPromptTemplate` defines the message structure once, with **placeholders** in `{curly braces}` that are filled in at run time:
- `{language}` – the target language (used in the system message)
- `{text}` – the text to translate (used in the user message)

In [51]:
from langchain_core.prompts import ChatPromptTemplate

generic_template = "Translate the following into {language}:"

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", generic_template),
        ("user", "{text}")
    ]
)

### Test the prompt template on its own
Invoking the template with a dictionary of values fills in the placeholders. The result is a `ChatPromptValue` — the ready-to-send list of `SystemMessage` + `HumanMessage` (no model call happens yet).

In [47]:
result = prompt.invoke({"language":"German","text":"I love you"})

In [48]:
result

ChatPromptValue(messages=[SystemMessage(content='Translate the following into German:', additional_kwargs={}, response_metadata={}), HumanMessage(content='I love you', additional_kwargs={}, response_metadata={})])

## Step 8: Full chain — prompt → model → parser
This is the complete LCEL pipeline:

```
{"language", "text"} ──► prompt ──► model ──► parser ──► translated string
```

Now any text can be translated into any language just by changing the input dictionary.

In [52]:
chain =  prompt | model | parser
chain.invoke({"language":"German","text":"I love you"})

'Ich liebe dich'

## Step 9: Reusing the same pattern for a different task — article generation
The same `prompt | model | parser` structure works for any task; only the template changes. Here the system prompt asks the model to write an article in a given `{language}` about a given `{topic}`, and the user message requests a detailed article.

(The output below is long, so it may be truncated in the notebook view.)

In [64]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

generic_template = """
Write an article in {language} about {topic}.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", generic_template),
    ("user", "Write a detailed and interesting article.")
])

parser = StrOutputParser()

chain = prompt | model | parser

response = chain.invoke({
    "language": "german",
    "topic": "Love"
})

print(response)

**Liebe – Das unsichtbare Band, das die Welt bewegt**

> „Liebe ist nicht das, was man erwartet, sondern das, was man bekommt.“ – **George Sand**

---

## 1. Einleitung: Warum wir über Liebe schreiben

Liebe ist das zentrale Thema aller menschlichen Erzählungen. Sie inspiriert Dichöre, Schriften, Kunstwerke, Filme und sogar Gesetzestexte. Doch was bedeutet Liebe eigentlich? Ist sie ein biologisches Bedürfnis, ein soziales Konstrukt oder ein metaphysisches Ideal? In diesem Artikel tauchen wir tief in die verschiedenen Facetten der Liebe ein – von klassischen philosophischen Begriffen über psychologische Theorien bis hin zu zeitgenössischen Herausforderungen in einer digitalisierten Welt.

---

## 2. Die klassischen Typen der Liebe

Der griechische Philosoph **Platon** unterschied bereits im 4. Jahrhundert v. Chr. fünf Grundformen der Liebe. Sie bilden bis heute die Basis für die Diskussion über romantische, freundschaftliche und selbstlose Bindungen.

| Typ | Name | Bedeutung | Beispiel

## Summary
- **Model** (`ChatGroq`) generates text from a list of messages.
- **Output parser** (`StrOutputParser`) turns the model's message into a plain string.
- **Prompt template** (`ChatPromptTemplate`) builds messages from reusable templates with placeholders.
- **LCEL** (`|`) connects these pieces into a single chain that runs with one `.invoke(...)` call.
- With **LangSmith tracing** enabled, each chain run can be inspected in the LangSmith dashboard.